# Easy multihop: building a candidate benchmark for intermediate readouts

**Question.** Can we construct short, elementary two-hop prompts for comparing
J-Lens and logit-lens visibility of an unspoken intermediate, without the
`Renault is a company from → France` continuation slot?

**Answer.** This notebook builds **100 base items, each with two phrasings (200
prompts)** across geography, animals, opposites, word forms, and arithmetic.
The exported file is `data/evaluations/lens-eval-multihop-easy.json`. The executed
checks below establish structural validity, literal-label absence, and GPT-2/Qwen
whole-token coverage. **Model answerability and lens discrepancies have not been
measured.** This is a candidate benchmark, not a model-filtered result.

The question concerns the *intermediate's rank under each lens*, not final-answer
accuracy. Final answers and separate hop questions are included for subsequent
model-specific eligibility diagnostics. Correct answers do not prove that the
annotated intermediate was used by the model.

## Construction and scoring contract

* Facts and elementary relations are curated below, not chosen using either lens.
  There are 32 geography, 20 animal, 20 opposite, 12 word-form, and 16 arithmetic
  base items. Some intermediates and answers recur; counts are not counts of
  independent concepts.
* Natural-language templates request the final property before describing the
  bridge. Arithmetic templates contain an unevaluated subexpression. The bridge
  and answer labels must both be absent as complete words/numbers in the prompt.
  This is a literal-leakage check, not a proof that no prefix predicts the bridge.
* `intermediates` contains one canonical bridge label. `target` is the final
  answer, used only for eligibility diagnostics. `hop1_prompt` and `hop2_prompt`
  are diagnostic prompts and must **not** be concatenated to the evaluation prompt.
* The primary readout is the **last prompt token**, before the final answer.
  Evaluate the intermediate under both lenses at exactly this same position and
  layer set. Report paired pass@k curves/AUC and rank differences. If exploring
  all positions, separately audit ordinary final-output bridge predictability;
  do not select examples on which the intermediate-layer logit lens fails.
* Aggregate the two phrasings within `base_id` before averaging base items. Report
  family-level scores as well as an equal-family macro average. Cluster
  uncertainty by `group_id` (shared bridge), which also keeps all phrasings and
  relations for one bridge in the same deterministic `dev`/`test` split.
* `dev`/`test` are for choosing evaluation settings, not fitting a lens. Hashing
  bridge labels selects about 20% of each family's unique bridges for development,
  independently of model/lens results. No model
  correctness, token rank, or lens advantage is used to discard items.
* Canonical labels follow the current evaluation loader; no new alias semantics
  are assumed. Use strict complete-token scoring and report unsupported labels
  for each tokenizer. Arithmetic intermediates use number words (Qwen splits
  multi-digit numerals), while arithmetic targets use digits. Neither is
  automatically expanded into synonyms under this dataset name.

Animal descriptions use ordinary textbook prototypes; opposites can admit other
valid words. This set tests annotated concept visibility and is not an exhaustive
semantic answer key. Some chains also permit direct shortcuts. The new wording
reduces an obvious continuation confound without claiming to eliminate all of them.

In [1]:
import hashlib
import json
import random
import re
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 180)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "jlens" / "evaluation.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
OUT_PATH = ROOT / "data/evaluations/lens-eval-multihop-easy.json"
SEED = 17
base_items = []


def add_base(base_id, family, relation, intermediate, target, prompts,
             hop1_prompt, hop2_prompt):
    """Record one independently curated chain and its two prompt templates."""
    base_items.append({
        "base_id": base_id, "family": family, "relation": relation,
        "intermediate": str(intermediate), "target": str(target),
        "prompts": prompts, "hop1_prompt": hop1_prompt,
        "hop2_prompt": hop2_prompt,
    })

## 1. Geography: capital → country → language or continent

Use familiar capitals, omit capitals that explicitly contain the country label,
and avoid transcontinental countries in the continent relation. Language labels
are conventional dominant languages, not claims that a country is monolingual.

In [2]:
# Country, capital, dominant language, continent.
geography = [
    ("France", "Paris", "French", "Europe"),
    ("Italy", "Rome", "Italian", "Europe"),
    ("Spain", "Madrid", "Spanish", "Europe"),
    ("Germany", "Berlin", "German", "Europe"),
    ("Portugal", "Lisbon", "Portuguese", "Europe"),
    ("Japan", "Tokyo", "Japanese", "Asia"),
    ("China", "Beijing", "Mandarin", "Asia"),
    ("Greece", "Athens", "Greek", "Europe"),
    ("Thailand", "Bangkok", "Thai", "Asia"),
    ("Vietnam", "Hanoi", "Vietnamese", "Asia"),
    ("Brazil", "Brasilia", "Portuguese", "South America"),
    ("Netherlands", "Amsterdam", "Dutch", "Europe"),
    ("Argentina", "Buenos Aires", "Spanish", "South America"),
    ("Sweden", "Stockholm", "Swedish", "Europe"),
    ("Poland", "Warsaw", "Polish", "Europe"),
    ("India", "New Delhi", "Hindi", "Asia"),
]
for country, capital, language, continent in geography:
    for relation, answer, prompts, hop2 in [
        ("country_language", language, [
            f"The most widely spoken language of the country whose capital is {capital} is",
            f"The most widely spoken language in the country with {capital} as its capital is",
        ], f"The most widely spoken language in {country} is"),
        ("country_continent", continent, [
            f"The continent containing the country whose capital is {capital} is",
            f"The continent where the country with {capital} as its capital is located is",
        ], f"The continent containing {country} is"),
    ]:
        add_base(f"{country.lower()}-{relation}", "geography", relation,
                 country, answer, prompts,
                 f"The country whose capital is {capital} is", hop2)

## 2. Animals: familiar description → animal → property

Clues describe the animal without naming it. Second hops ask for legs, offspring,
sounds, or products. Number answers are digits to keep the canonical scoring
contract explicit; valid number words may require separate eligibility grading.

In [3]:
# Intermediate, indirect description, relation, answer.
animals = [
    ("dog", "the domestic animal that barks", "legs", "4"),
    ("dog", "the domestic animal that barks", "young", "puppy"),
    ("cat", "the domestic animal that says meow", "legs", "4"),
    ("cat", "the domestic animal that says meow", "young", "kitten"),
    ("cow", "the farm animal that says moo", "drink", "milk"),
    ("cow", "the farm animal that says moo", "young", "calf"),
    ("sheep", "the farm animal that says baa", "fiber", "wool"),
    ("sheep", "the farm animal that says baa", "young", "lamb"),
    ("horse", "the farm animal ridden by jockeys", "legs", "4"),
    ("horse", "the farm animal ridden by jockeys", "sound", "neigh"),
    ("hen", "the female farm bird that clucks", "food", "eggs"),
    ("bee", "the striped flying insect that lives in a hive", "food", "honey"),
    ("bee", "the striped flying insect that lives in a hive", "legs", "6"),
    ("spider", "the animal that spins silk webs", "legs", "8"),
    ("ant", "the small insect that marches in trails to picnic food", "legs", "6"),
    ("butterfly", "the colorful winged insect that develops from a caterpillar", "legs", "6"),
    ("elephant", "the large land animal with a trunk and tusks", "legs", "4"),
    ("snake", "the long reptile that hisses and slithers", "legs", "0"),
    ("duck", "the water bird that says quack", "legs", "2"),
    ("penguin", "the flightless black-and-white bird of Antarctica", "legs", "2"),
]
animal_templates = {
    "legs": ["The number of legs on {clue} is", "The leg count of {clue} is"],
    "young": ["The young of {clue} is called a", "The name for a baby of {clue} is"],
    "drink": ["The drink produced by {clue} is", "The drink obtained from {clue} is"],
    "fiber": ["The textile fiber obtained from {clue} is", "The fiber shorn from {clue} is"],
    "food": ["The food produced by {clue} is", "The food obtained from {clue} is"],
    "sound": ["The characteristic sound made by {clue} is a", "The characteristic call of {clue} is a"],
}
for animal, clue, relation, answer in animals:
    templates = animal_templates[relation]
    add_base(f"{animal}-{relation}", "animals", f"animal_{relation}", animal,
             answer, [t.format(clue=clue) for t in templates],
             f"The name of {clue} is",
             templates[0].format(clue=f"the {animal}"))

## 3. Opposites: familiar property → adjective → antonym

Only the intended adjective is the intermediate. For example, the annotation for
the temperature of ice is `cold`, and the answer is `hot`. The answer is not
counted as an intermediate.

In [4]:
opposites = [
    ("ice", "the temperature of ice", "cold", "hot"),
    ("boiling-water", "the temperature of boiling water", "hot", "cold"),
    ("elephant-size", "the size of an elephant", "big", "small"),
    ("mouse-size", "the size of a mouse", "small", "big"),
    ("anvil", "the weight of an anvil", "heavy", "light"),
    ("feather", "the weight of a feather", "light", "heavy"),
    ("racing-car", "the speed of a racing car", "fast", "slow"),
    ("snail", "the speed of a snail", "slow", "fast"),
    ("soaked-towel", "the condition of a towel soaked in water", "wet", "dry"),
    ("desert-sand", "the condition of sand in a rainless desert", "dry", "wet"),
    ("stone", "the firmness of a stone", "hard", "soft"),
    ("pillow", "the firmness of a fluffy pillow", "soft", "hard"),
    ("bottle", "a bottle with nothing inside", "empty", "full"),
    ("glass", "a glass filled to the brim", "full", "empty"),
    ("doorway", "a door swung wide to let people through", "open", "closed"),
    ("shut-door", "a door that has been shut", "closed", "open"),
    ("mountaintop", "the altitude of a mountaintop", "high", "low"),
    ("valley", "the altitude of a valley floor", "low", "high"),
    ("giraffe-neck", "the length of a giraffe's neck", "long", "short"),
    ("cropped-hair", "the length of closely cropped hair", "short", "long"),
]
for key, clue, adjective, answer in opposites:
    add_base(f"{key}-opposite", "opposites", "property_antonym", adjective,
             answer, [
                 f"The opposite of the adjective describing {clue} is",
                 f"An antonym of the word describing {clue} is",
             ], f"The adjective describing {clue} is",
             f"The opposite of {adjective} is")

## 4. Word forms: description → singular noun → plural

Irregular plurals make the intermediate and final answer visibly different.
These are lexical composition items, reported separately from factual relations.

In [5]:
word_forms = [
    ("child", "a young human being", "children"),
    ("man", "an adult male human", "men"),
    ("woman", "an adult female human", "women"),
    ("foot", "the body part that goes inside a shoe", "feet"),
    ("tooth", "one of the hard white structures used for biting", "teeth"),
    ("mouse", "the small rodent that squeaks", "mice"),
    ("person", "an individual human being", "people"),
    ("leaf", "one of the flat green parts growing on a tree", "leaves"),
    ("wolf", "the wild canine that howls in packs", "wolves"),
    ("knife", "the sharp utensil used to slice bread", "knives"),
    ("shelf", "a horizontal board on a wall used to hold books", "shelves"),
    ("life", "the state of being alive", "lives"),
]
for singular, clue, plural in word_forms:
    add_base(f"{singular}-plural", "word_forms", "description_plural", singular,
             plural, [
                 f"The plural of the noun meaning {clue} is",
                 f"The plural form of the word for {clue} is",
             ], f"The singular noun meaning {clue} is",
             f"The plural of {singular} is")

## 5. Arithmetic: parenthesized sum/difference → value → product

All intermediate values and final answers are computed, not hand-labeled. These
items are an elementary arithmetic stratum; success on them by a small model
must still be measured. No intermediate equals a displayed operand.
Bridge labels use number words so each has a whole-token spelling in both audited
tokenizers. Canonical digit targets can be unsupported for next-token diagnostics
on Qwen; that does not prevent scoring the intermediate. These scores concern the
word spelling, not the union of digit and word spellings.

In [6]:
arithmetic = [
    (2, "+", 3, 2), (3, "+", 4, 2), (4, "+", 5, 2), (2, "+", 4, 3),
    (3, "+", 5, 2), (4, "+", 6, 2), (5, "+", 6, 2), (5, "+", 7, 2),
    (8, "-", 3, 2), (9, "-", 2, 2), (9, "-", 3, 2), (9, "-", 5, 3),
    (8, "-", 2, 3), (7, "-", 3, 2), (9, "-", 6, 4), (8, "-", 5, 2),
]
number_words = {
    3: "three", 4: "four", 5: "five", 6: "six", 7: "seven", 8: "eight",
    9: "nine", 10: "ten", 11: "eleven", 12: "twelve",
}
for a, op, b, multiplier in arithmetic:
    intermediate = a + b if op == "+" else a - b
    target = intermediate * multiplier
    assert intermediate not in (a, b, multiplier, target)
    expression = f"({a} {op} {b}) * {multiplier}"
    operation = "plus" if op == "+" else "minus"
    add_base(f"{a}-{operation}-{b}-times-{multiplier}", "arithmetic",
             f"{operation}_multiply", number_words[intermediate], target, [
                 f"{expression} =",
                 f"The result of multiplying ({a} {op} {b}) by {multiplier} is",
             ], f"{a} {op} {b} =", f"{intermediate} * {multiplier} =")

## 6. Deterministic export and structural checks

The standard `name`, `prompt`, `target`, and `intermediates` fields are sufficient
for existing evaluators. Additional metadata describes grouping and diagnostics.
No global default dataset list is modified. The fixed shuffle also avoids
identical bridge labels in the legacy evaluator's half-list-offset controls;
those controls remain a coarse baseline, not family-matched distractors.

In [7]:
def contains_label(text, label):
    """Find whole labels, including numeric labels, case-insensitively."""
    return bool(re.search(r"(?<!\w)" + re.escape(label) + r"(?!\w)",
                          text, flags=re.IGNORECASE))


dev_groups = set()
for family in sorted({base["family"] for base in base_items}):
    groups = {base["intermediate"].casefold() for base in base_items
              if base["family"] == family}
    ordered = sorted(groups, key=lambda word: hashlib.sha256(
        f"{SEED}:{word}".encode()).hexdigest())
    dev_groups.update(ordered[:max(1, round(len(ordered) / 5))])

items = []
for base in base_items:
    bridge = base["intermediate"]
    group_id = bridge.casefold()
    for template_id, prompt in enumerate(base["prompts"]):
        items.append({
            "name": f"{base['base_id']}-v{template_id + 1}",
            "prompt": prompt, "target": base["target"],
            "intermediates": [bridge], "base_id": base["base_id"],
            "family": base["family"], "relation": base["relation"],
            "template_id": template_id + 1, "group_id": group_id,
            "split": "dev" if group_id in dev_groups else "test",
            "hop1_prompt": base["hop1_prompt"],
            "hop2_prompt": base["hop2_prompt"],
        })

rng = random.Random(SEED)
for _ in range(10000):
    rng.shuffle(items)
    if all(item["intermediates"] != items[(i + len(items) // 2) % len(items)]["intermediates"]
           for i, item in enumerate(items)):
        break
else:
    raise RuntimeError("Could not find an order with distinct legacy controls")

assert len(base_items) == 100 and len(items) == 200
assert len({b["base_id"] for b in base_items}) == len(base_items)
assert len({i["name"] for i in items}) == len(items)
assert len({i["prompt"] for i in items}) == len(items)
for item in items:
    assert item["prompt"] == item["prompt"].strip()
    assert item["target"].casefold() != item["intermediates"][0].casefold()
    for label in [item["target"], *item["intermediates"]]:
        assert not contains_label(item["prompt"], label), (item["name"], label)
frame = pd.DataFrame(items)
assert frame.groupby("base_id").size().eq(2).all()
assert frame.groupby("group_id")["split"].nunique().eq(1).all()
assert frame.groupby("family")["split"].nunique().eq(2).all()

dataset = {
    "metadata": {
        "version": 1,
        "status": "unfiltered_candidates",
        "purpose": "Compare J-Lens and logit-lens ranks of unspoken intermediates",
        "readout_position": "last_prompt_token",
        "model_validation": "not_run",
        "lens_selection": "none",
        "n_base_items": len(base_items), "n_prompts": len(items),
        "shuffle_seed": SEED,
        "split_rule": "Per family, sort unique casefolded bridges by sha256('17:' + bridge); first max(1, round(n / 5)) enter dev; union dev groups across families",
        "builder": "notebooks/model_agnostic/multihop_easy/multihop_easy_build.ipynb",
    },
    "items": items,
}
OUT_PATH.write_text(json.dumps(dataset, indent=2, ensure_ascii=False) + "\n")
assert json.loads(OUT_PATH.read_text()) == dataset
print(f"Wrote {len(items)} prompts from {len(base_items)} base items.")
print(f"Distinct bridge labels: {frame.group_id.nunique()}")
print("All structural, literal-leakage, grouping, and legacy-control checks passed.")
display(frame.groupby("family", sort=False).agg(
    base_items=("base_id", "nunique"), prompts=("name", "size"),
    bridges=("group_id", "nunique"), targets=("target", "nunique")))
display(frame.groupby(["family", "split"]).base_id.nunique().unstack(fill_value=0))
display(frame[frame.template_id.eq(1)].groupby("family", sort=False).head(2)[
    ["family", "prompt", "intermediates", "target"]])

Wrote 200 prompts from 100 base items.
Distinct bridge labels: 72
All structural, literal-leakage, grouping, and legacy-control checks passed.


,base_items,prompts,bridges,targets
family,,,,
animals,20,40,14,14
geography,32,64,16,17
word_forms,12,24,12,12
opposites,20,40,20,20
arithmetic,16,32,10,10


split,dev,test
family,,
animals,4,16
arithmetic,3,13
geography,6,26
opposites,4,16
word_forms,2,10


,family,prompt,intermediates,target
1,geography,The continent containing the country whose capital is Stockholm is,[Sweden],Europe
2,animals,The number of legs on the domestic animal that barks is,[dog],4
3,geography,The most widely spoken language of the country whose capital is Athens is,[Greece],Greek
4,word_forms,The plural of the noun meaning the sharp utensil used to slice bread is,[knife],knives
5,opposites,The opposite of the adjective describing a bottle with nothing inside is,[empty],full
11,animals,The number of legs on the water bird that says quack is,[duck],2
12,arithmetic,(2 + 3) * 2 =,[five],10
16,opposites,The opposite of the adjective describing the firmness of a stone is,[hard],soft
19,word_forms,The plural of the noun meaning a horizontal board on a wall used to hold books is,[shelf],shelves
32,arithmetic,(5 + 6) * 2 =,[eleven],22


## 7. Tokenizer coverage (no model weights)

This is an audit only: it neither filters nor changes the dataset. The first run
may download tokenizer files; subsequent runs use the Hugging Face cache. Set
`AUDIT_TOKENIZERS = []` for an entirely offline rebuild without tokenizer files,
or add the ID of the small Qwen you plan to evaluate. Coverage is tokenizer-specific.
The audit uses the repository's strict decoded whole-token spelling lookup.

In [8]:
from transformers import AutoTokenizer

from jlens.strict_scoring import DecodedSpellings

AUDIT_TOKENIZERS = ["openai-community/gpt2-xl", "Qwen/Qwen3-0.6B"]
coverage_rows = []
for model_id in AUDIT_TOKENIZERS:
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    lookup = DecodedSpellings(tokenizer, vocab_size=len(tokenizer))
    labels = sorted({i["target"] for i in items} | {i["intermediates"][0] for i in items})
    supported = {word: bool(lookup(word, False)) for word in labels}
    for kind, words in [
        ("intermediate", [i["intermediates"][0] for i in items]),
        ("target", [i["target"] for i in items]),
    ]:
        missing = sorted({word for word in words if not supported[word]})
        coverage_rows.append({"tokenizer": model_id, "kind": kind,
                              "supported_items": sum(supported[word] for word in words),
                              "total_items": len(words), "unsupported_labels": missing})
display(pd.DataFrame(coverage_rows))
print("Coverage is lexical support, not evidence of model answerability.")

,tokenizer,kind,supported_items,total_items,unsupported_labels
0,openai-community/gpt2-xl,intermediate,200,200,[]
1,openai-community/gpt2-xl,target,196,200,[South America]
2,Qwen/Qwen3-0.6B,intermediate,200,200,[]
3,Qwen/Qwen3-0.6B,target,168,200,"[10, 12, 14, 16, 18, 20, 22, 24, South America]"


Coverage is lexical support, not evidence of model answerability.


## Using the dataset

In `notebooks/model_agnostic/model_agnostic_lens_dataset.ipynb`, select it explicitly:

```python
evals = {"multihop-easy": load_eval(REPO_DIR, "multihop-easy")}
```

The existing non-poetry readout rule selects the final prompt token. Extra fields
are ignored by the loader/scorer; join `base_id`, `family`, `group_id`, and `split`
back by item name for grouped reporting. Arithmetic bridge words and digit targets
here are not expanded by the `order-ops` synonym policy. For strict single-token answer
diagnostics, unsupported targets must stay unscored, not be credited on a first
token. Model-specific eligibility can additionally accept curated complete-answer
aliases, but such grading must be recorded separately from intermediate ranks.

Before claiming this is easy for a model, batch the composed, hop-1, and hop-2
prompts by length, right-pad, and read each last real token. Cache diagnostics
using a hash of the dataset, model revision, dtype, tokenizer, and BOS/chat format.
Report both the full candidate set and a clearly identified answerable subset;
never filter on either lens's intermediate ranks. Bootstrap shared bridge groups
for paired lens comparisons rather than treating 200 phrasings as independent.

## Conclusions

The file is reproducibly generated in the existing evaluation schema, with 100
base items and 200 phrasings, explicit bridge labels, two diagnostic hop prompts,
and grouped development/test splits. All 200 intermediate annotations have
whole-token spellings under GPT-2 and Qwen3-0.6B. GPT-2 supports 196/200 canonical
targets; Qwen supports 168/200. The remaining targets are multi-token numbers
and/or `South America`; exclude them only from strict next-token target diagnostics,
not from the intermediate-rank comparison. Primary evaluation is intermediate visibility
at the final prompt position. Literal absence is verified; semantic shortcuts and
prefix predictability have not been ruled out. No small-model answerability or
J-Lens advantage is asserted by this construction notebook.